# SAE-feature construct plots (`main_by_sae_features.py`)

Pulls results from WandB project `experiment_1_set_consistency_construct_paper`.
Three mix modes: positive, signed, negative decoder-direction perturbations.

Gemma grids are ordered: **16k big → medium → small, 262k big → medium → small**.
Medium SAEs have limited layers (subset only); row labels include the SAE release name.

In [ ]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from matplotlib.axes import Axes
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


# ── Constants ────────────────────────────────────────────────────────────────
LABEL_FONTSIZE = 15
LEGEND_FONTSIZE = 10
TITLE_FONTSIZE = 14
TICK_FONTSIZE = 12

import os
import wandb as _wandb_for_entity
# W&B entity: $WANDB_ENTITY if set, otherwise your default entity.
ENTITY = os.environ.get("WANDB_ENTITY") or _wandb_for_entity.Api().default_entity
PROJECT_CONSTRUCT = "appendix_interference"  # wandb.project in configs/appendix_interference/base.yaml
SETUP_NAME = "A_fixed_base"

MIX_MODES = [("rand_pos10", "a)"), ("rand_signed10", "b)"), ("rand_neg10", "c)")]
MIX_MODE_TITLES = {
    "rand_pos10":    "Positive perturbations",
    "rand_signed10": "Signed perturbations",
    "rand_neg10":    "Negative perturbations",
}

# k value used for the layer-vs-lost-rate inlay inside each panel
B_C_MODE: Any = 1501

OUTPUT_DIR = Path("wandb_sae_features_grids")
CACHE_DIR  = Path(".wandb_sae_features_cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = [
    "gemma-3-270m",
    "gemma-3-1b-pt",
    "gemma-3-4b-pt",
    "gemma-3-12b-pt",
    "gemma-3-27b-pt",
    "gpt2-small",
    "meta-llama/Llama-3.1-8B",
    "mistralai/Mistral-7B-v0.1",
]

GEMMA_MODELS = {
    "gemma-3-270m",
    "gemma-3-1b-pt",
    "gemma-3-4b-pt",
    "gemma-3-12b-pt",
    "gemma-3-27b-pt",
}
# Per-model interpretability layers (the subset available for medium SAEs).
# All SAE types for a given model are shown on these layers so color coding is
# consistent across rows. Non-medium SAEs additionally show layer 0.
GEMMA_INTERPRETABILITY_LAYERS: Dict[str, List[int]] = {
    "gemma-3-270m":  [5, 9, 12, 15],
    "gemma-3-1b-pt": [7, 13, 17, 22],
    "gemma-3-4b-pt": [9, 17, 22, 29],
    "gemma-3-12b-pt": [12, 24, 31, 41],
    "gemma-3-27b-pt": [16, 31, 40, 53],
}

EXCLUDE_FROM_MAIN = {("gemma-3-270m", "16k big")}
SPECIAL_CASES    = [("gemma-3-270m", "16k big")]


# ── Shared utility helpers ────────────────────────────────────────────────────
def cfg_get(cfg: Any, *keys: str, default=None):
    if not isinstance(cfg, dict):
        return default
    cur, ok = cfg, True
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            ok = False
            break
    return cur if ok else cfg.get(".".join(keys), default)


def safe_name(x: str) -> str:
    return x.replace("/", "__").replace(" ", "_")


def latest_run(runs):
    return sorted(runs, key=lambda r: str(getattr(r, "created_at", "")))[-1]


def model_sort_key(x: str) -> int:
    return MODEL_ORDER.index(x) if x in MODEL_ORDER else 10**9


def get_run_name(run) -> str:
    return str(getattr(run, "name", "") or "")


def sparse_integer_xticks(vals: List[int], n_ticks: int = 5) -> List[int]:
    vals = sorted(map(int, vals))
    if len(vals) <= n_ticks:
        return vals
    idxs = [round(i * (len(vals) - 1) / (n_ticks - 1)) for i in range(n_ticks)]
    ticks, seen = [], set()
    for idx in idxs:
        x = vals[idx]
        if x not in seen:
            ticks.append(x)
            seen.add(x)
    return ticks


def pretty_model_name(model_name: str) -> str:
    return {
        "gemma-3-270m":               "Gemma 3 270M",
        "gemma-3-1b-pt":              "Gemma 3 1B",
        "gemma-3-4b-pt":              "Gemma 3 4B",
        "gemma-3-12b-pt":             "Gemma 3 12B",
        "gemma-3-27b-pt":             "Gemma 3 27B",
        "gpt2-small":                 "GPT-2 Small",
        "meta-llama/Llama-3.1-8B":    "Llama 3.1 8B",
        "mistralai/Mistral-7B-v0.1":  "Mistral 7B",
    }.get(model_name, model_name)


def get_model_name(run) -> Optional[str]:
    cfg   = run.config or {}
    group = str(getattr(run, "group", "") or "")

    vals = [
        cfg_get(cfg, "sae", "model", "name"),
        cfg_get(cfg, "model", "name"),
        cfg_get(cfg, "sae_model_name"),
        cfg_get(cfg, "model_name"),
        group,
    ]
    for v in vals:
        if v is None:
            continue
        s = str(v)
        if s in MODEL_ORDER:
            return s
        s_low = s.lower()
        if "gemma" in s_low and "270m" in s_low:  return "gemma-3-270m"
        if "gemma" in s_low and "1b"   in s_low:  return "gemma-3-1b-pt"
        if "gemma" in s_low and "4b"   in s_low:  return "gemma-3-4b-pt"
        if "gemma" in s_low and "12b"  in s_low:  return "gemma-3-12b-pt"
        if "gemma" in s_low and "27b"  in s_low:  return "gemma-3-27b-pt"
        if "gpt2-small" in s_low:                 return "gpt2-small"
        if "llama-3.1-8b" in s_low or ("meta-llama" in s_low and "8b" in s_low):
            return "meta-llama/Llama-3.1-8B"
        if "mistral-7b" in s_low:                 return "mistralai/Mistral-7B-v0.1"

    blob_parts = [get_run_name(run), group]
    try:
        blob_parts.append(json.dumps(cfg, sort_keys=True))
    except Exception:
        blob_parts.append(str(cfg))
    blob = " || ".join(blob_parts).lower()

    if "gemma-scope-2-270m" in blob or ("270m" in blob and "gemma" in blob): return "gemma-3-270m"
    if "gemma-scope-2-1b"   in blob or ("1b"   in blob and "gemma" in blob): return "gemma-3-1b-pt"
    if "gemma-scope-2-4b"   in blob or ("4b"   in blob and "gemma" in blob): return "gemma-3-4b-pt"
    if "gemma-scope-2-12b"  in blob or ("12b"  in blob and "gemma" in blob): return "gemma-3-12b-pt"
    if "gemma-scope-2-27b"  in blob or ("27b"  in blob and "gemma" in blob): return "gemma-3-27b-pt"
    if "gpt2-small" in blob:                                                  return "gpt2-small"
    if "llama-3.1-8b" in blob or ("meta-llama" in blob and "8b" in blob):    return "meta-llama/Llama-3.1-8B"
    if "mistral-7b" in blob:                                                  return "mistralai/Mistral-7B-v0.1"
    return None


def get_sae_short_label(run) -> Optional[str]:
    run_name = get_run_name(run).lower()
    if re.search(r"gpt2-small-resid-post-v\d+-128k", run_name): return "128k"
    if re.search(r"gpt2-small-resid-post-v\d+-32k",  run_name): return "32k"
    if re.search(r"llama[_ ]scope[_ ]lxr[_ ]8x[_ ]l\{\}r[_ ]8x",  run_name): return "32k"
    if re.search(r"llama[_ ]scope[_ ]lxr[_ ]32x[_ ]l\{\}r[_ ]32x", run_name): return "128k"  # 2**17 = 131,072 latents; called 128k by the release, like GPT-2 v5-128k
    if "mistral-7b-res-wg" in run_name:                         return "65k"  # d_sae = 65,536

    cfg = run.config or {}
    vals = [
        cfg_get(cfg, "sae", "release"),
        cfg_get(cfg, "sae_release"),
        cfg_get(cfg, "sae", "name"),
        cfg_get(cfg, "sae_name"),
        cfg.get("sae"),
        run_name,
        # id_template checked last: fixes runs whose run_name doesn't embed the
        # size/variant (e.g. 1B 16k medium), without disturbing runs that already
        # resolve correctly via run_name (e.g. 12B).
        cfg_get(cfg, "sae", "id_template"),
        cfg_get(cfg, "sae.id_template"),
    ]
    for v in vals:
        if v is None:
            continue
        s = str(v).lower()
        for pat in [
            r"(\d+k)_(big|small|medium|canonical)\.yaml",
            r"(\d+k)_(big|small|medium|canonical)",
            r"width[_ -](\d+k)(?:[_ -]l\d+)?[_ -](big|small|medium|canonical)\b",
            r"\b(\d+k)\s*[_ -]?\s*(big|small|medium|canonical)\b",
        ]:
            m = re.search(pat, s)
            if m:
                return f"{m.group(1)} {m.group(2)}"
        if "base.yaml" in s or re.search(r"\bbase\b", s):
            return "base"

    raw = (
        cfg_get(cfg, "sae", "release")
        or cfg_get(cfg, "sae_release")
        or cfg_get(cfg, "sae", "name")
        or cfg_get(cfg, "sae_name")
    )
    return str(raw).replace(".yaml", "").replace("_", " ").strip() if raw is not None else run_name


def get_sae_release_name(run) -> Optional[str]:
    """Return the full SAE release string from the run config (e.g. 'gemma-scope-2-12b-pt-res')."""
    cfg = run.config or {}
    for keys in [("sae", "release"), ("sae_release",), ("sae", "name"), ("sae_name",)]:
        v = cfg_get(cfg, *keys)
        if v is not None:
            s = str(v)
            if s and not s.startswith("{"):
                return s
    return None


def select_representative_layers(layers: List[int], n_select: int = 5) -> List[int]:
    layers = sorted(map(int, layers))
    if len(layers) <= n_select:
        return layers
    idxs = [round(i * (len(layers) - 1) / (n_select - 1)) for i in range(n_select)]
    out, seen = [], set()
    for idx in idxs:
        l = layers[idx]
        if l not in seen:
            out.append(l)
            seen.add(l)
    return out


def is_medium_sae(sae_label: str) -> bool:
    return bool(re.search(r"\bmedium\b", sae_label.lower()))


def select_gemma_layers(available_layers: List[int], sae_label: str, model_name: str) -> List[int]:
    """Select layers for Gemma SAE plots so color coding is consistent across SAE types.

    All SAE types for a model plot the model's interpretability layers in the same
    order, giving them identical colors. Non-medium SAEs get layer 0 appended so it
    doesn't displace the color assignment of the shared interpretability layers.
    """
    available = set(available_layers)
    interp_layers = GEMMA_INTERPRETABILITY_LAYERS.get(model_name, [])
    interp = [l for l in interp_layers if l in available]
    if not is_medium_sae(sae_label) and 0 in available:
        interp = interp + [0]
    return interp


def sae_sort_key(x: str):
    if x == "base":
        return (999, 999, x)
    m = re.match(r"(\d+)k(?:\s+(big|small|medium))?$", x)
    if m:
        size, var = int(m.group(1)), m.group(2)
        return (size, 0 if var == "big" else 1 if var == "medium" else 2, x)
    return (998, 998, x)


def parse_table_json(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8") as f:
        obj = json.load(f)
    return pd.DataFrame(obj["data"], columns=obj["columns"])


def find_table_file_by_parts(run, *parts: str):
    hits = []
    for f in run.files():
        name = f.name
        if all(p in name for p in parts) and name.endswith(".table.json"):
            hits.append(f)
    return sorted(hits, key=lambda x: x.name)[0] if hits else None


def download_file(file_obj, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    # WandB downloads to out_dir/file_obj.name (preserving subdirectory structure).
    # Check this full path to avoid redundant network downloads.
    local_path = out_dir / file_obj.name
    if local_path.exists():
        return local_path
    return Path(file_obj.download(root=str(out_dir), replace=False).name)


def add_panel_letter(ax, letter: str) -> None:
    ax.text(
        -0.18, 1.01, letter,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=18, fontweight="bold",
        clip_on=False,
    )


def draw_blank(ax, subtitle: str = "nothing found") -> None:
    ax.text(0.5, 0.56, subtitle, ha="center", va="center", transform=ax.transAxes, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_alpha(0.3)


# ── Construct-specific data helpers ──────────────────────────────────────────
def normalize_construct_df(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    if "mean_value" in df.columns and "mean" not in df.columns: rename["mean_value"] = "mean"
    if "std_value"  in df.columns and "std"  not in df.columns: rename["std_value"]  = "std"
    if "metric"     in df.columns and "metric_name" not in df.columns: rename["metric"] = "metric_name"
    if "mix_mode"   in df.columns and "pooling"     not in df.columns: rename["mix_mode"] = "pooling"
    df = df.rename(columns=rename).copy()
    if "setup_name" not in df.columns: df["setup_name"] = SETUP_NAME
    if "pooling"    not in df.columns: df["pooling"]    = ""
    df["layer"] = df["layer"].astype(int)
    df["k"]     = df["k"].astype(int)
    df["mean"]  = df["mean"].astype(float)
    df["std"]   = df["std"].astype(float)
    df["setup_name"] = df["setup_name"].astype(str).str.lstrip("_")
    df["pooling"]    = df["pooling"].astype(str)
    if "metric_name" in df.columns:
        df["metric_name"] = df["metric_name"].astype(str).str.replace(" ", "_", regex=False)
    return df


def load_construct_metric_df(run, metric_name: str, mix_mode: str) -> Optional[pd.DataFrame]:
    f = find_table_file_by_parts(run, mix_mode, f"{metric_name}_data")
    if f is None:
        return None
    path = download_file(
        f,
        CACHE_DIR / safe_name(get_model_name(run) or "unknown_model") / safe_name(get_sae_short_label(run) or "unknown_sae"),
    )
    return normalize_construct_df(parse_table_json(path))


def df_to_plot_dicts_construct(df: pd.DataFrame, mix_mode: str, metric_name: str = "lost_rate"):
    df = df.copy()
    df["setup_name"] = df["setup_name"].astype(str).str.lstrip("_")
    if "metric_name" in df.columns:
        df["metric_name"] = df["metric_name"].astype(str).str.replace(" ", "_", regex=False)
    df = df[df["setup_name"] == SETUP_NAME]
    if "pooling" in df.columns:
        df = df[df["pooling"] == mix_mode]
    if "metric_name" in df.columns:
        df = df[df["metric_name"] == metric_name]

    mean_by_layer_by_k: Dict[int, Dict[int, float]] = defaultdict(dict)
    std_by_layer_by_k:  Dict[int, Dict[int, float]] = defaultdict(dict)
    for _, row in df.iterrows():
        l, k = int(row["layer"]), int(row["k"])
        mean_by_layer_by_k[l][k] = float(row["mean"])
        std_by_layer_by_k[l][k]  = float(row["std"])
    return dict(mean_by_layer_by_k), dict(std_by_layer_by_k), sorted(mean_by_layer_by_k.keys())


def construct_df_to_layer_dicts(df: pd.DataFrame, mix_mode: str, mode: Any):
    df = df.copy()
    df["setup_name"] = df["setup_name"].astype(str).str.lstrip("_")
    if "metric_name" in df.columns:
        df["metric_name"] = df["metric_name"].astype(str).str.replace(" ", "_", regex=False)
    df = df[df["setup_name"] == SETUP_NAME]
    if "pooling" in df.columns:
        df = df[df["pooling"] == mix_mode]
    if "metric_name" in df.columns:
        df = df[df["metric_name"] == "lost_rate"]

    target_k = 1 if mode == "aggregated" else int(mode)
    df = df[df["k"] == target_k].copy()
    if df.empty:
        raise ValueError(f"no rows for k={target_k}")

    layers = sorted(df["layer"].astype(int).tolist())
    mean_by_layer = {int(r["layer"]): float(r["mean"]) for _, r in df.iterrows()}
    std_by_layer  = {int(r["layer"]): float(r["std"])  for _, r in df.iterrows()}
    return layers, mean_by_layer, std_by_layer


def _sparse_layer_xticks(layers: List[int], n_ticks: int = 5) -> List[int]:
    layers = sorted(map(int, layers))
    if len(layers) <= n_ticks:
        return layers
    idxs = [round(i * (len(layers) - 1) / (n_ticks - 1)) for i in range(n_ticks)]
    ticks, seen = [], set()
    for idx in idxs:
        x = layers[idx]
        if x not in seen:
            ticks.append(x)
            seen.add(x)
    return ticks


def construct_xticks() -> List[int]:
    return [1, 500, 1000, 1500, 2000]


# ── Plotting primitives ───────────────────────────────────────────────────────
def plot_against_k_on_ax(
    ax: Axes,
    selected_layers: List[int],
    mean_by_layer_by_k: Dict[int, Dict[int, float]],
    std_by_layer_by_k: Optional[Dict[int, Dict[int, float]]] = None,
    title: str = "",
    ylabel: str = "",
    ylim: Optional[Tuple[float, float]] = (-0.02, 1.02),
    show_legend: bool = True,
    legend_loc: str = "upper right",
) -> None:
    layers = [int(x) for x in selected_layers]

    mean_int: Dict[int, Dict[int, float]] = {
        int(lk): {int(k): float(v) for k, v in d.items()}
        for lk, d in mean_by_layer_by_k.items()
    }
    std_int: Dict[int, Dict[int, float]] = {}
    if std_by_layer_by_k is not None:
        std_int = {
            int(lk): {int(k): float(v) for k, v in d.items()}
            for lk, d in std_by_layer_by_k.items()
        }

    ks = sorted({k for layer in layers for k in mean_int.get(layer, {}).keys()})
    if not ks:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        return

    for layer in layers:
        x = ks
        y = [mean_int.get(layer, {}).get(k, float("nan")) for k in ks]
        s = [std_int.get(layer, {}).get(k, 0.0) for k in ks] if std_by_layer_by_k is not None else []

        line = ax.plot(x, y, linewidth=2.0, label=f"layer {layer}")
        color = line[0].get_color()
        if s:
            ax.fill_between(x, [yy - ss for yy, ss in zip(y, s)],
                               [yy + ss for yy, ss in zip(y, s)],
                            color=color, alpha=0.18, linewidth=0)

    ax.set_xlabel(r"step $k$", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.grid(True, linewidth=0.5, alpha=0.25)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_yticks(np.linspace(0, 1, 11))
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, loc=legend_loc)


def plot_against_k_with_inlay_on_ax(
    ax: Axes,
    *,
    selected_layers: List[int],
    mean_by_layer_by_k: Dict[int, Dict[int, float]],
    std_by_layer_by_k: Dict[int, Dict[int, float]],
    mean_inlay_by_layer: Dict[int, float],
    std_inlay_by_layer: Dict[int, float],
    inlay_layers: List[int],
    title: str = "",
    ylabel: str = "",
    ylim: Tuple[float, float] = (-0.02, 1.02),
    show_legend: bool = True,
    inlay_loc: str = "upper left",
    inlay_width: str = "24%",
    inlay_height: str = "24%",
    inlay_bbox_to_anchor: Tuple[float, float, float, float] = (0.15, 0.02, 1.0, 1.0),
    legend_loc: str = "upper right",
) -> None:
    plot_against_k_on_ax(
        ax=ax,
        selected_layers=selected_layers,
        mean_by_layer_by_k=mean_by_layer_by_k,
        std_by_layer_by_k=std_by_layer_by_k,
        title=title,
        ylabel=ylabel,
        ylim=ylim,
        show_legend=False,
    )

    x_in = [int(l) for l in inlay_layers]
    y_in = [float(mean_inlay_by_layer[l]) for l in x_in]
    s_in = [float(std_inlay_by_layer[l])  for l in x_in]

    axins = inset_axes(
        ax,
        width=inlay_width, height=inlay_height,
        loc=inlay_loc,
        bbox_to_anchor=inlay_bbox_to_anchor,
        bbox_transform=ax.transAxes,
        borderpad=0.8,
    )
    axins.patch.set_facecolor("white")
    axins.patch.set_alpha(0.72)

    line_in = axins.plot(x_in, y_in, marker="o", markersize=3, linewidth=1.6)
    color_in = line_in[0].get_color()
    axins.fill_between(
        x_in,
        [yy - ss for yy, ss in zip(y_in, s_in)],
        [yy + ss for yy, ss in zip(y_in, s_in)],
        color=color_in, alpha=0.18, linewidth=0,
    )

    inlay_xticks = _sparse_layer_xticks(x_in, n_ticks=5)
    axins.set_xticks(inlay_xticks)
    axins.set_xlim(min(x_in) - 0.5, max(x_in) + 0.5)
    axins.set_yticks(np.linspace(0, 1, 6))
    axins.set_ylim(-0.02, 1.02)
    axins.tick_params(axis="both", labelsize=max(9, TICK_FONTSIZE - 3), pad=2)
    axins.grid(True, alpha=0.25)
    axins.margins(x=0.06, y=0.08)

    if show_legend:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles, labels, fontsize=LEGEND_FONTSIZE, loc=legend_loc,
                  ncol=2, columnspacing=0.8, handlelength=1.4)


# ── Panel / row rendering ─────────────────────────────────────────────────────
def get_row_label(run, sae_label: str) -> str:
    """Two-line label for medium SAEs: '<width> medium\n(<release>)'."""
    if not is_medium_sae(sae_label) or run is None:
        return sae_label
    release = get_sae_release_name(run)
    if release:
        return f"{sae_label}\n({release})"
    return sae_label


def plot_construct_panel(
    ax: Axes,
    run,
    mix_mode: str,
    show_legend: bool,
    last_row: bool,
    sae_label: str = "",
    special_case: bool = False,
) -> None:
    if run is None:
        draw_blank(ax)
        return
    try:
        df = load_construct_metric_df(run, "lost_rate", mix_mode)
        if df is None or df.empty:
            draw_blank(ax)
            return

        mean_d, std_d, layers = df_to_plot_dicts_construct(df, mix_mode=mix_mode)
        if not layers:
            draw_blank(ax)
            return

        model_name = get_model_name(run)
        if model_name in GEMMA_MODELS:
            selected_layers = select_gemma_layers(layers, sae_label, model_name)
        else:
            selected_layers = select_representative_layers(layers, n_select=5)

        try:
            inlay_layers, mean_inlay, std_inlay = construct_df_to_layer_dicts(
                df, mix_mode=mix_mode, mode=B_C_MODE
            )
        except Exception:
            inlay_layers, mean_inlay, std_inlay = [], {}, {}

        _bottom_models = {"gpt2-small", "meta-llama/Llama-3.1-8B"}
        _use_bottom = model_name in _bottom_models and mix_mode == "rand_pos10"
        _inlay_loc  = "lower left"  if _use_bottom else "upper left"
        _legend_loc = "lower right" if _use_bottom else "upper right"

        if inlay_layers:
            plot_against_k_with_inlay_on_ax(
                ax=ax,
                selected_layers=selected_layers,
                mean_by_layer_by_k={l: mean_d[l] for l in selected_layers},
                std_by_layer_by_k={l: std_d[l]  for l in selected_layers},
                mean_inlay_by_layer=mean_inlay,
                std_inlay_by_layer=std_inlay,
                inlay_layers=inlay_layers,
                title="",
                ylabel=r"lost rate $\mathcal{R}$",
                ylim=(-0.02, 1.02),
                show_legend=show_legend,
                inlay_loc=_inlay_loc,
                inlay_width="24%",
                inlay_height="24%",
                inlay_bbox_to_anchor=(0.15, 0.02, 1.0, 1.0),
                legend_loc=_legend_loc,
            )
        else:
            plot_against_k_on_ax(
                ax=ax,
                selected_layers=selected_layers,
                mean_by_layer_by_k={l: mean_d[l] for l in selected_layers},
                std_by_layer_by_k={l: std_d[l]  for l in selected_layers},
                title="",
                ylabel=r"lost rate $\mathcal{R}$",
                ylim=(-0.02, 1.02),
                show_legend=show_legend,
                legend_loc=_legend_loc,
            )

        label_fs = 16 if special_case else 14
        tick_fs  = 13 if special_case else 12
        ax.set_ylabel(r"lost rate $\mathcal{R}$", fontsize=label_fs)
        ax.tick_params(axis="both", labelsize=tick_fs)
        ax.set_xticks(construct_xticks())
        ax.set_xlim(0, 2020)
        ax.set_ylim(-0.02, 1.02)
        ax.set_yticks([i / 10 for i in range(11)])
        if last_row:
            ax.set_xlabel(r"step $k$", fontsize=label_fs)
        else:
            ax.set_xlabel("")
            ax.tick_params(axis="x", labelbottom=False)

    except Exception as e:
        print(f"[construct:{mix_mode}] -> {e}")
        draw_blank(ax)


def plot_construct_row(
    axs,
    run,
    sae_label: str,
    row_label: str,
    last_row: bool,
    show_letters: bool,
    show_legend: bool,
    special_case: bool = False,
) -> None:
    for j, (mix_mode, letter) in enumerate(MIX_MODES):
        ax = axs[j]
        if j == 0:
            # Row label on the left; two-line for medium SAEs
            multiline = "\n" in row_label
            ax.text(
                -0.22, 0.5, row_label,
                transform=ax.transAxes,
                rotation=90,
                ha="center", va="center",
                fontsize=10 if multiline else 12,
                fontweight="bold",
                clip_on=False,
            )
        if show_letters:
            add_panel_letter(ax, letter)
        plot_construct_panel(
            ax, run,
            mix_mode=mix_mode,
            sae_label=sae_label,
            show_legend=(show_legend and j == 0),
            last_row=last_row,
            special_case=special_case,
        )


def _set_column_titles(axes) -> None:
    """Overwrite axes titles on the first row with human-readable mix mode names."""
    for j, (mix_mode, _) in enumerate(MIX_MODES):
        axes[0, j].set_title(MIX_MODE_TITLES.get(mix_mode, mix_mode),
                             fontsize=TITLE_FONTSIZE, pad=6)


# ── WandB data collection ─────────────────────────────────────────────────────
def collect_construct_runs():
    api  = wandb.Api(timeout=60)
    runs = list(api.runs(f"{ENTITY}/{PROJECT_CONSTRUCT}"))
    grouped: Dict[Tuple[str, str], list] = defaultdict(list)
    for run in runs:
        model_name = get_model_name(run)
        sae_label  = get_sae_short_label(run)
        if model_name is None or sae_label is None:
            continue
        grouped[(model_name, sae_label)].append(run)
    return {k: latest_run(v) for k, v in grouped.items()}


# ── Grid generators ───────────────────────────────────────────────────────────
def make_gemma_grids(runs_construct: dict) -> None:
    """
    One figure per Gemma model.  SAE rows are ordered:
    16k big, 16k medium, 16k small, 262k big, 262k medium, 262k small.
    Medium rows show the SAE release name as a second line in the row label.
    """
    model_to_saes: Dict[str, List[str]] = defaultdict(list)
    for model_name, sae_label in runs_construct.keys():
        if model_name not in GEMMA_MODELS:
            continue
        if (model_name, sae_label) not in EXCLUDE_FROM_MAIN:
            model_to_saes[model_name].append(sae_label)

    for model_name in sorted(model_to_saes.keys(), key=model_sort_key):
        sae_labels = sorted(set(model_to_saes[model_name]), key=sae_sort_key)
        print(f"  {pretty_model_name(model_name)}: {sae_labels}")
        n_rows = len(sae_labels)
        n_cols = len(MIX_MODES)

        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(5.2 * n_cols, max(2.8 * n_rows, 4.3)),
            squeeze=False,
        )
        fig.suptitle(pretty_model_name(model_name), fontsize=20, y=0.995)

        for i, sae_label in enumerate(sae_labels):
            run       = runs_construct.get((model_name, sae_label))
            row_label = get_row_label(run, sae_label)
            plot_construct_row(
                axes[i], run,
                sae_label=sae_label,
                row_label=row_label,
                last_row=(i == n_rows - 1),
                show_letters=(i == 0),
                show_legend=(i == 0),
            )

        _set_column_titles(axes)
        fig.subplots_adjust(left=0.14, right=0.995, bottom=0.06, top=0.90,
                            hspace=0.15, wspace=0.24)
        out_path = OUTPUT_DIR / f"{safe_name(model_name)}__construct_grid.png"
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        plt.close(fig)
        print(f"  wrote {out_path}")


def make_construct_grids(runs_construct: dict) -> None:
    """One figure per model (all models, same SAE ordering)."""
    model_to_saes: Dict[str, List[str]] = defaultdict(list)
    for model_name, sae_label in runs_construct.keys():
        if (model_name, sae_label) not in EXCLUDE_FROM_MAIN:
            model_to_saes[model_name].append(sae_label)

    for model_name in sorted(model_to_saes.keys(), key=model_sort_key):
        sae_labels = sorted(set(model_to_saes[model_name]), key=sae_sort_key)
        n_rows = len(sae_labels)
        n_cols = len(MIX_MODES)

        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(5.2 * n_cols, max(2.8 * n_rows, 4.3)),
            squeeze=False,
        )
        fig.suptitle(pretty_model_name(model_name), fontsize=20, y=0.985)

        for i, sae_label in enumerate(sae_labels):
            run       = runs_construct.get((model_name, sae_label))
            row_label = get_row_label(run, sae_label)
            plot_construct_row(
                axes[i], run,
                sae_label=sae_label,
                row_label=row_label,
                last_row=(i == n_rows - 1),
                show_letters=(i == 0),
                show_legend=(i == 0),
            )

        _set_column_titles(axes)
        fig.subplots_adjust(left=0.11, right=0.995, bottom=0.06, top=0.86,
                            hspace=0.15, wspace=0.24)
        out_path = OUTPUT_DIR / f"{safe_name(model_name)}__construct_grid.png"
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        plt.close(fig)
        print(f"  wrote {out_path}")


def make_construct_special_case_figures(runs_construct: dict) -> None:
    for model_name, sae_label in SPECIAL_CASES:
        run = runs_construct.get((model_name, sae_label))
        if run is None:
            continue

        n_modes = len(MIX_MODES)
        fig, axes = plt.subplots(n_modes, 1, figsize=(7.2, 7.4), squeeze=False)

        for i, (mix_mode, letter) in enumerate(MIX_MODES):
            ax = axes[i, 0]
            add_panel_letter(ax, letter)
            plot_construct_panel(
                ax, run,
                mix_mode=mix_mode,
                sae_label=sae_label,
                show_legend=(i == 0),
                last_row=(i == n_modes - 1),
                special_case=True,
            )

        # Column title on the top panel
        axes[0, 0].set_title(MIX_MODE_TITLES.get(MIX_MODES[0][0], MIX_MODES[0][0]),
                             fontsize=TITLE_FONTSIZE)

        fig.subplots_adjust(left=0.12, right=0.99, bottom=0.08, top=0.95, hspace=0.22)
        out_path = OUTPUT_DIR / f"{safe_name(model_name)}__{safe_name(sae_label)}__construct_single_vertical.png"
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        plt.close(fig)
        print(f"  wrote {out_path}")


In [ ]:
runs_construct = collect_construct_runs()
print("models:", sorted({k[0] for k in runs_construct.keys()}, key=model_sort_key))
print("SAE labels:", sorted({k[1] for k in runs_construct.keys()}, key=sae_sort_key))

In [ ]:
# ── Debug: inspect raw extraction for 1b and 12b runs ────────────────────────
api = wandb.Api(timeout=60)
raw_runs = list(api.runs(f"{ENTITY}/{PROJECT_CONSTRUCT}"))

print(f"Total runs in project: {len(raw_runs)}\n")

print("=== Collected grouped labels (all models) ===")
_dbg_grouped: dict = {}
for run in raw_runs:
    mn = get_model_name(run)
    sl = get_sae_short_label(run)
    key = (mn, sl)
    _dbg_grouped[key] = _dbg_grouped.get(key, 0) + 1
for (mn, sl), cnt in sorted(_dbg_grouped.items(), key=lambda x: (str(x[0][0]), str(x[0][1]))):
    flag = " *** MEDIUM?" if sl == "16k medium" else ""
    print(f"  model={mn!r:30s}  sae_label={sl!r:40s}  count={cnt}{flag}")

for target_str, label in [("1b", "1B"), ("12b", "12B")]:
    print(f"\n=== Runs with '{target_str}' in run.name (raw) ===")
    for run in raw_runs:
        if target_str not in get_run_name(run).lower():
            continue
        mn  = get_model_name(run)
        sl  = get_sae_short_label(run)
        cfg = run.config or {}
        sae_cfg = cfg.get("sae") if isinstance(cfg.get("sae"), dict) else {}
        sae_release  = sae_cfg.get("release")     if sae_cfg else cfg.get("sae.release")
        sae_id_tpl   = sae_cfg.get("id_template") if sae_cfg else cfg.get("sae.id_template")
        group = str(getattr(run, "group", "") or "")
        print(f"  run.name={run.name!r}")
        print(f"    group={group!r}  -> model={mn!r}  sae_label={sl!r}")
        print(f"    sae.release={sae_release!r}  sae.id_template={sae_id_tpl!r}")
        print()

In [ ]:
print("=== Gemma grids (16k big/medium/small, 262k big/medium/small) ===")
make_gemma_grids(runs_construct)

In [ ]:
print("=== All-model grids ===")
make_construct_grids(runs_construct)

In [ ]:
print("=== Special-case single figures ===")
make_construct_special_case_figures(runs_construct)